# Compartment - mmBERT pipeline (Kaggle)

Chạy toàn bộ M3 chain trong một session: **probe → warmup → train5 → predict**.

### Lấy repo (2 cách, ưu tiên mount trước)
1. **Kaggle dataset mount**: repo nằm dưới `/kaggle/input/**/Compartment` (đoạn auto-detect). **Lưu ý**: dataset đang mount là code cũ — phải re-upload bản mới nhất (M3) mới có `mm/`.
2. **Git clone từ GitHub**: nếu không thấy path mount được, notebook clone `REPO_URL` (mặc định `AmnO-O/MoTune`, nhánh `main`) vào `/kaggle/working/Compartment`. **Bắt buộc push trước**: GitHub hiện vẫn ở commit `9e272b0` (trước M1, không có `mm/`).
- Repo **private**: đặt `GITHUB_TOKEN` (PAT) dưới dạng `os.environ['GITHUB_TOKEN']` trước khi chạy cell setup.

### Knobs (sửa dòng `MODE` bên dưới)
- `MODE = 'all'` chạy cả chain; đặt `probe | warmup | train5 | predict` để chạy riêng.
- `WARMUP_EPOCHS`, `WARMUP_BATCH` (hạ xuống 16 nếu OOM ở phase MLM).
- `MLM_DATA`: các file context dùng cho MLM warmup.
- `TRAIN5_SET`: thêm `--set` cho train5 (cách nhau bằng dấu phẩy).

Outputs ở `/kaggle/working`: `models/`, `submission/`, `metrics.json`, `history.json`, `oof_predictions.npz`, `trial_metrics.json`.

In [1]:
import glob, os, subprocess, sys
from pathlib import Path

def _importable(name: str) -> bool:
    try:
        __import__(name)
        return True
    except Exception:
        return False

# ---- repo source -----------------------------------------------------------
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/AmnO-O/MoTune.git')
REPO_BRANCH = os.environ.get('REPO_BRANCH', 'main')
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')    # set for private repos

# ---- always run code from a fresh GitHub clone ----------------------------
# The Kaggle-mount snapshot is frozen when the dataset is uploaded, so running
# code from the mount silently executes stale buggy versions. Data/trial are
# still read from the mount: mm.utils._find_data_dir resolves the mount dataset/
# dir via /kaggle/input/datasets/ieltsmater/compartment/Compartment, and
# _resolve_trial_path looks next to it (.../Compartment/trial).
dest = Path('/kaggle/working/Compartment')
url = REPO_URL
if GITHUB_TOKEN:
    url = url.replace('https://', f'https://{GITHUB_TOKEN}@')
dest.parent.mkdir(parents=True, exist_ok=True)
if (dest / 'run.py').is_file():
    print('refreshing existing clone at', dest)
    subprocess.run(['git', '-C', str(dest), 'fetch', '--depth', '1', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(dest), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)
else:
    print('cloning', REPO_URL)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, url, str(dest)], check=True)
REPO = dest
os.chdir(REPO)
print('repo:', REPO)
print('head:', subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'],
                              capture_output=True, text=True).stdout.strip())

# ---- knobs ----------------------------------------------------------------
MODE = os.environ.get('MODE', 'all')            # all | probe | warmup | train5 | predict
WARMUP_EPOCHS = int(os.environ.get('WARMUP_EPOCHS', '12'))
WARMUP_BATCH = os.environ.get('WARMUP_BATCH', '64')   # e.g. 16 if OOM in MLM phase
MLM_DATA = 'en-nn-train.tsv,de-nn-train.tsv,en-pv-train.tsv,de-pv-train.tsv,nctti_en.tsv'
TRAIN5_SET = os.environ.get('TRAIN5_SET', '')       # extra --set flags, comma-separated

# ---- tunable config overrides ('' = keep run.py default) -----------------
# Edit values here to tune a run without touching --set flags below.
OVERRIDES = {
    'warmup_batch_size': WARMUP_BATCH,      # e.g. 16 if OOM in MLM warmup
    'warmup_lr': '',
    'mlm_mask_span': '',                    # mask | span | both
    'mlm_mask_prob': '',
    'mlm_random_prob': '',
    'freeze_epochs': '',
    'lora_epochs': '',
    'lora_rank': '',
    'lora_alpha': '',
    'lora_from_layer': '18',                # LoRA window: top layers (0 = all)
    'batch_size': '',
    'head_lr': '',
    'encoder_lr': '',
    'ccc_weight': '',
    'lambda_rank': '',
    'lambda_compound': '',
    'num_workers': '',
}

def cfg_sets(*extra):
    out = []
    for k, v in OVERRIDES.items():
        if v not in (None, ''):
            out += ['--set', f'{k}={v}']
    for e in extra:
        out += ['--set', e]
    return out


# ---- memory ------------------------------------------------------------------
# expandable segments reduce T4 fragmentation; inherited by the run.py subprocess
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

# ---- ensure imports exist (Kaggle already ships them) ---------------------
for m in ('torch', 'transformers', 'pandas', 'numpy', 'sklearn', 'scipy'):
    if not _importable(m):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', m], check=False)

def run(args):
    print('\n>>>', ' '.join(args))
    subprocess.run(args, cwd=REPO, check=True)


cloning https://github.com/AmnO-O/MoTune.git


Cloning into '/kaggle/working/Compartment'...


repo: /kaggle/working/Compartment
head: 5180401 update


In [2]:
# --- M2 gate: real-tokenizer alignment + MLM masking sanity ---------------
if MODE in ('all', 'probe'):
    run([sys.executable, 'run.py', 'probe'] + cfg_sets())



>>> /usr/bin/python3 run.py probe --set lora_from_layer=18
08:43:21 | INFO    | Using device: cuda
08:43:21 | INFO    | GPU: Tesla T4 | 15.6 GB
08:43:21 | INFO    | Working data dir: /kaggle/input/datasets/ieltsmater/compartment/Compartment/dataset
08:43:21 | INFO    | Output dir: /kaggle/working
08:43:21 | INFO    | Config:
{
  "mode": "train5",
  "seed": 42,
  "debug_cuda": false,
  "backbone": "jhu-clsp/mmBERT-base",
  "hidden_size": 768,
  "head_pool": "attn",
  "context_pool": "mean+cls",
  "head_mode": "reg",
  "num_bins": 6,
  "head_hidden": 128,
  "dropout": 0.2,
  "use_lm_features": false,
  "data_path": null,
  "output_dir": null,
  "max_context_length": 256,
  "max_mlm_length": 128,
  "train_file": "en-nn-train.tsv",
  "trial_file": "en-nn-trial.tsv",
  "aux_data_paths": [],
  "mlm_data_paths": [],
  "warmup_mlm_epochs": 0,
  "mlm_mask_span": "both",
  "mlm_mask_prob": 0.8,
  "mlm_random_prob": 0.1,
  "warmup_batch_size": 32,
  "warmup_lr": 5e-05,
  "warmup_warmup_ratio": 0

08:43:34 | INFO    | en-nn-train.tsv: 3480 rows / 444 compounds
08:43:36 | INFO    | CompDataset: 3480 rows, 98% aligned, 1 degenerate, 3480 labeled
08:43:36 | INFO    | alignment en  :  98% aligned (3432/3480) | mod-only 3432 | head-only 3432 | degenerate 1
08:43:36 | INFO    | alignment all :  98% aligned (3432/3480) | mod-only 3432 | head-only 3432 | degenerate 1
08:43:36 | INFO    | alignment all : aligned=3432 (99%) degenerate=1
08:43:36 | INFO    | probe row 0 mod  toks=[6]  text=<bos> ▁Correspondence , ▁letter books , [▁account] ▁books , ▁and ▁writings ▁together ▁with ▁family ▁papers ▁concerning ▁Charles , ▁Clement , ▁Edward , ▁James , ▁Nicholas , ▁Thomas , ▁and ▁William ▁S . ▁Bid dle . <eos>
08:43:36 | INFO    | probe row 0 head toks=[7]  text=<bos> ▁Correspondence , ▁letter books , ▁account [▁books] , ▁and ▁writings ▁together ▁with ▁family ▁papers ▁concerning ▁Charles , ▁Clement , ▁Edward , ▁James , ▁Nicholas , ▁Thomas , ▁and ▁William ▁S . ▁Bid dle . <eos>
08:43:36 | INFO    |

In [3]:
# --- Phase 0: compound-aware MLM warmup + LoRA merge ----------------------
# Writes /kaggle/working/models/warmup_merged.pt (used as base by train5).
if MODE in ('all', 'warmup'):
    args = [sys.executable, 'run.py', 'warmup'] + cfg_sets(
        f'warmup_mlm_epochs={WARMUP_EPOCHS}',
        f'mlm_data_paths={MLM_DATA}')
    run(args)



>>> /usr/bin/python3 run.py warmup --set lora_from_layer=18 --set warmup_mlm_epochs=8 --set mlm_data_paths=en-nn-train.tsv,de-nn-train.tsv,en-pv-train.tsv,de-pv-train.tsv,nctti_en.tsv
08:43:41 | INFO    | Using device: cuda
08:43:41 | INFO    | GPU: Tesla T4 | 15.6 GB
08:43:41 | INFO    | Working data dir: /kaggle/input/datasets/ieltsmater/compartment/Compartment/dataset
08:43:41 | INFO    | Output dir: /kaggle/working
08:43:41 | INFO    | Config:
{
  "mode": "train5",
  "seed": 42,
  "debug_cuda": false,
  "backbone": "jhu-clsp/mmBERT-base",
  "hidden_size": 768,
  "head_pool": "attn",
  "context_pool": "mean+cls",
  "head_mode": "reg",
  "num_bins": 6,
  "head_hidden": 128,
  "dropout": 0.2,
  "use_lm_features": false,
  "data_path": null,
  "output_dir": null,
  "max_context_length": 256,
  "max_mlm_length": 128,
  "train_file": "en-nn-train.tsv",
  "trial_file": "en-nn-trial.tsv",
  "aux_data_paths": [],
  "mlm_data_paths": [
    "en-nn-train.tsv",
    "de-nn-train.tsv",
    "en-p

Loading weights:  86%|████████▌ | 118/138 [00:00<00:00, 3022.44it/s, Materializing param=model.layers.18.mlp.Wi.weight]      

08:44:01 | INFO    | en-nn-train.tsv: 3480 warmup rows
08:44:01 | INFO    | de-nn-train.tsv: 3298 warmup rows
08:44:02 | INFO    | en-pv-train.tsv: 1557 warmup rows
08:44:02 | INFO    | de-pv-train.tsv: 1490 warmup rows
08:44:02 | INFO    | nctti_en.tsv: 539 warmup rows
08:44:02 | INFO    | total warmup rows: 10364
08:44:10 | INFO    | warmup rows: 10364 (324 batches/epoch)
08:44:10 | INFO    | warmup LoRA matched 10 modules (layers >= 18), e.g. ['lm.model.layers.18.attn.Wqkv', 'lm.model.layers.18.attn.Wo', 'lm.model.layers.19.attn.Wqkv'] (auto-fell back to targets ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'attn.Wqkv', 'attn.Wo', 'attn.linear'])
08:44:10 | INFO    | warmup trainable: 24 unique parameter tensors (20 LoRA, 4 MLM Head)
08:44:10 | INFO    | MLM head chosen: ModernBertPredictionHead (head), decoder=decoder, vocab=256000
08:45:22 | INFO    | warmup epoch 1/8 | MLM loss 2.9468
08:46:20 | INFO    | warmup epoch 2/8 | MLM loss 2.3311
08:47:20 | INFO    | warmup epoch 3/8 | MLM l

In [4]:
import os

# Set môi trường trực tiếp trong Python để tránh deadlock GPU và DataLoader
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# --- Phase 1: 5-fold CV (compound-grouped, no leak) + OOF -----------------
if MODE in ('all', 'train80'):
    args = [
        sys.executable, 'run.py', 'train80', 
        '--set', 'num_workers=0',            # Chống deadlock Kaggle
        '--set', 'lora_from_layer=18',       # LoRA từ layer 18
        '--set', 'batch_size=64',            # Batch size 64 mượt hơn cho Phase 1
        '--set', 'freeze_epochs=20',          # Hạ từ 20 xuống 5 epoch
        '--set', 'head_lr=2e-4',             # Head Learning Rate
        '--set', 'predict_mode=single',       # Sửa lỗi: chọn '5fold' hoặc 'single'
    ] + cfg_sets()

    if TRAIN5_SET:
        args += [x.strip() for x in TRAIN5_SET.split(',') if x.strip()]
        
    run(args)


>>> /usr/bin/python3 run.py train5 --set num_workers=0 --set lora_from_layer=18
08:52:24 | INFO    | Using device: cuda
08:52:24 | INFO    | GPU: Tesla T4 | 15.6 GB
08:52:24 | INFO    | Working data dir: /kaggle/input/datasets/ieltsmater/compartment/Compartment/dataset
08:52:24 | INFO    | Output dir: /kaggle/working
08:52:24 | INFO    | Config:
{
  "mode": "train5",
  "seed": 42,
  "debug_cuda": false,
  "backbone": "jhu-clsp/mmBERT-base",
  "hidden_size": 768,
  "head_pool": "attn",
  "context_pool": "mean+cls",
  "head_mode": "reg",
  "num_bins": 6,
  "head_hidden": 128,
  "dropout": 0.2,
  "use_lm_features": false,
  "data_path": null,
  "output_dir": null,
  "max_context_length": 256,
  "max_mlm_length": 128,
  "train_file": "en-nn-train.tsv",
  "trial_file": "en-nn-trial.tsv",
  "aux_data_paths": [],
  "mlm_data_paths": [],
  "warmup_mlm_epochs": 0,
  "mlm_mask_span": "both",
  "mlm_mask_prob": 0.8,
  "mlm_random_prob": 0.1,
  "warmup_batch_size": 32,
  "warmup_lr": 5e-05,
  "wa

CalledProcessError: Command '['/usr/bin/python3', 'run.py', 'train5', '--set', 'num_workers=0', '--set', 'lora_from_layer=18']' returned non-zero exit status 1.

In [ ]:
# --- Predict trial (5-fold ensemble) + submission -------------------------
if MODE in ('all', 'predict'):
    run([sys.executable, 'run.py', 'predict'] + cfg_sets())


In [ ]:
import json
import pandas as pd

work = Path('/kaggle/working')
for name in ('metrics.json', 'trial_metrics.json'):
    p = work / name
    if p.is_file():
        print(f'--- {name} ---')
        print(json.dumps(json.loads(p.read_text(encoding='utf-8')), indent=2))

sub = work / 'submission' / 'en-nn-trial-pred.tsv'
if sub.is_file():
    df = pd.read_csv(sub, sep='\t', header=None, names=['tID', 'Modifier', 'Head'])
    print('\n--- submission (en-nn-trial-pred.tsv, no header) ---')
    print(df.head())
    print('rows:', len(df))

### Nộp submission
1. File `en-nn-trial-pred.tsv` ở `/kaggle/working/submission/`.
2. Download rồi nộp ở challenge (3 cột `tID, Modifier, Head`, không header).

### Lưu ý
- **trial ρ là artifact n=2** — đừng dùng trial rho làm tín hiệu; xem **OOF ρ** (mean của `Mod`/`Head`) trong `metrics.json`.
- Muốn chạy lại chain với config khác: chỉnh `TRAIN5_SET`, ví dụ `TRAIN5_SET='--set,batch_size=16,--set,freeze_epochs=2'`. WARMUP thay đổi thì phải chạy lại `warmup` + `train5` + `predict` trong cùng session (ckpt ở `/kaggle/working` không tồn tại giữa các session).